In [3]:
import jax
import jax.numpy as jnp
from jax import lax, jit, vmap
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time

# --- Configuration (Optimized for Stability & Speed)
MAX_RECURSION_DEPTH    = 1_000_000   # Maximum recursion depth.
OPTIMAL_DEPTH_STEP     = 250_000     # Base branch step length.
DIMENSIONAL_CONSTRAINT = 0.8
BATCH_SIZE             = 50_000_000  # 50M samples per batch.

# Clamp bounds to avoid overflow/NaNs.
VAL_CLAMP_LOW  = -100.0
VAL_CLAMP_HIGH =  100.0

# -------------------------------------------------------------------------
# 1) Dynamic pi & phi functions (with stabilization)
# -------------------------------------------------------------------------
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@jit
def stabilize_depth(depth):
    return depth / (1 + jnp.log1p(depth + 1))

# -------------------------------------------------------------------------
# 2) Base function: 250K-step linear recursion.
# -------------------------------------------------------------------------
@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=OPTIMAL_DEPTH_STEP, scale_factor=1.0):
    depth = stabilize_depth(jnp.minimum(depth, MAX_RECURSION_DEPTH))
    def body_fn(i, val):
        pi_dyn  = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale   = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT
        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val
    return lax.fori_loop(0, depth.astype(jnp.int32), body_fn, x)

# -------------------------------------------------------------------------
# 3) Branch-Recycling Functions:
#    a) branch_recycle: Replicate x into branches, process each branch for branch_depth,
#       then average the outputs.
#    b) process_with_branches: Apply branch_recycle repeatedly until total_depth is reached.
# -------------------------------------------------------------------------
def branch_recycle(x, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5):
    xs = jnp.stack([x] * num_branches, axis=0)  # Shape: (num_branches, BATCH_SIZE)
    branch_fn = vmap(lambda xi: dppu_with_dynamic_pi_phi(xi, depth=branch_depth, scale_factor=scale_factor))
    branch_outputs = branch_fn(xs)
    return jnp.mean(branch_outputs, axis=0)

def process_with_branches(x, total_depth, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5):
    iterations = total_depth // branch_depth
    for _ in range(iterations):
        x = branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)
    return x

# -------------------------------------------------------------------------
# 4) Multi-core CPU Sharding Setup using an 8-core mesh.
# -------------------------------------------------------------------------
devices = jax.devices("cpu")
if len(devices) < 8:
    devices = mesh_utils.create_device_mesh((8,))
else:
    devices = devices[:8]
mesh = Mesh(devices, ("data",))
sharding = PositionalSharding(mesh.devices.flat)

batch_input = jnp.linspace(0, 10, BATCH_SIZE)
batch_input = jax.device_put(batch_input, sharding)

# -------------------------------------------------------------------------
# 5) Wrap branch_recycle in a pjit so that each branch step is distributed.
#    Mark total_depth, num_branches, branch_depth, scale_factor as static.
# -------------------------------------------------------------------------
@partial(pjit,
         static_argnames=("num_branches", "branch_depth", "scale_factor"),
         in_shardings=(sharding,),  # Only the non-static argument x is sharded.
         out_shardings=sharding)
def branch_recycle_pjit(x, num_branches, branch_depth, scale_factor):
    return branch_recycle(x, num_branches=num_branches, branch_depth=branch_depth, scale_factor=scale_factor)

# -------------------------------------------------------------------------
# 6) Process with Branch Recycling Verbosely.
#    This function repeatedly calls the pjit-wrapped branch_recycle and prints benchmark info.
# -------------------------------------------------------------------------
def process_with_branches_verbose(x, total_depth, num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP, scale_factor=0.5, print_every=1):
    iterations = total_depth // branch_depth
    timings = []
    for i in range(iterations):
        start = time.time()
        x = branch_recycle_pjit(x, num_branches, branch_depth, scale_factor)
        jax.block_until_ready(x)
        elapsed = time.time() - start
        timings.append(elapsed)
        if (i + 1) % print_every == 0:
            mean_val = float(jnp.mean(x))
            print(f"After iteration {i+1}/{iterations}: mean(x)={mean_val:.6f}, chunk time: {elapsed:.6f} sec")
    return x, timings

# -------------------------------------------------------------------------
# 7) Benchmarking Function: Run and collect intermediate benchmarks.
# -------------------------------------------------------------------------
def run_benchmarks_verbose(batch_input, depths=(250_000, 500_000, 1_000_000),
                           num_branches=2, branch_depth=OPTIMAL_DEPTH_STEP,
                           scale_factor=0.5, num_trials=2, print_every=1):
    results = []
    for depth in depths:
        print(f"\n--- Running benchmark for total_depth={depth} ---")
        for trial in range(num_trials):
            start_time = time.time()
            final_x, timings = process_with_branches_verbose(batch_input, depth, num_branches, branch_depth, scale_factor, print_every)
            jax.block_until_ready(final_x)
            elapsed = time.time() - start_time
            mean_val = float(jnp.mean(final_x))
            results.append({"depth": depth, "trial": trial, "time": elapsed, "mean_output": mean_val, "chunk_timings": timings})
            print(f"Trial {trial}: Final mean(x)={mean_val:.6f}, total time: {elapsed:.6f} sec")
    return results

# -------------------------------------------------------------------------
# 8) Main Script.
# -------------------------------------------------------------------------
if __name__ == "__main__":
    # Precompile the base function to avoid on-the-fly compilation.
    _ = dppu_with_dynamic_pi_phi(jnp.ones((BATCH_SIZE,)), depth=OPTIMAL_DEPTH_STEP)

    # Run verbose branch recycling benchmarks.
    bench_results = run_benchmarks_verbose(
        batch_input,
        depths=[250_000, 500_000, 1_000_000],
        num_branches=2,
        branch_depth=OPTIMAL_DEPTH_STEP,
        scale_factor=0.5,
        num_trials=2,
        print_every=1
    )

    print("\nCollected Benchmark Results:")
    for r in bench_results:
        print(r)




--- Running benchmark for total_depth=250000 ---
After iteration 1/1: mean(x)=nan, chunk time: 313.346241 sec
Trial 0: Final mean(x)=nan, total time: 313.456126 sec
After iteration 1/1: mean(x)=nan, chunk time: 129.487417 sec
Trial 1: Final mean(x)=nan, total time: 129.491871 sec

--- Running benchmark for total_depth=500000 ---
After iteration 1/2: mean(x)=nan, chunk time: 129.487362 sec
After iteration 2/2: mean(x)=nan, chunk time: 129.487397 sec
Trial 0: Final mean(x)=nan, total time: 258.982449 sec
After iteration 1/2: mean(x)=nan, chunk time: 129.487352 sec
After iteration 2/2: mean(x)=nan, chunk time: 129.487424 sec
Trial 1: Final mean(x)=nan, total time: 258.981967 sec

--- Running benchmark for total_depth=1000000 ---
After iteration 1/4: mean(x)=nan, chunk time: 129.487546 sec
After iteration 2/4: mean(x)=nan, chunk time: 129.487697 sec
After iteration 3/4: mean(x)=nan, chunk time: 129.487719 sec
After iteration 4/4: mean(x)=nan, chunk time: 129.487777 sec
Trial 0: Final mean